# Uncertainty Quantification: MC Dropout and Temperature Scaling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ludwig-ai/ludwig/blob/main/examples/uncertainty/uncertainty.ipynb)

Modern deep learning classifiers are often **overconfident** — they assign extreme probabilities (near 0 or 1) even when they should be uncertain. This notebook demonstrates two complementary techniques for addressing that:

| Technique | What it does | When to use |
|---|---|---|
| **Temperature Scaling** | Post-hoc calibration: rescales logits by a learned scalar so probabilities match empirical frequencies | You want trustworthy probability outputs for ranking, risk scoring, or downstream decisions |
| **MC Dropout** | Runs multiple stochastic forward passes at inference time to produce per-sample uncertainty estimates | You want to flag high-uncertainty predictions for human review or active learning |

**Dataset**: UCI Wine Quality (red wine) — 1,599 samples, 11 physicochemical features, binary target: quality ≥ 7 is "good".

**References**:
- Guo et al., *On Calibration of Modern Neural Networks*, ICML 2017
- Gal & Ghahramani, *Dropout as a Bayesian Approximation*, ICML 2016

In [ ]:
!pip install ludwig --quiet

## Dataset

We use the [UCI Wine Quality dataset](https://archive.ics.uci.edu/ml/datasets/wine+quality) (red wine). The original quality score ranges from 3–8; we binarise it: **quality ≥ 7 → good (1), otherwise bad (0)**.

The dataset has ~14% positive examples, making probability calibration especially important.

In [ ]:
import logging
import os
import shutil
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

WINE_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "wine-quality/winequality-red.csv"
)

df = pd.read_csv(WINE_URL, sep=";")
df.columns = [c.replace(" ", "_") for c in df.columns]
df["quality"] = (df["quality"] >= 7).astype(int)

print(f"Dataset shape: {df.shape}")
print(f"Positive class (quality>=7): {df['quality'].mean():.1%}")
df.head()

## Calibration Utilities

We implement Expected Calibration Error (ECE) and reliability diagrams from scratch so we can compare models without extra dependencies.

In [ ]:
def expected_calibration_error(probabilities, labels, n_bins=10):
    """Compute ECE: weighted average of |confidence - accuracy| per bin."""
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(probabilities)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probabilities >= lo) & (probabilities < hi)
        if mask.sum() == 0:
            continue
        conf = probabilities[mask].mean()
        acc = labels[mask].mean()
        ece += mask.sum() / n * abs(conf - acc)
    return float(ece)


def plot_reliability_diagram(probabilities_dict, labels, n_bins=10, title="Reliability Diagram"):
    """Plot reliability diagrams for multiple models side by side."""
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0, 1], [0, 1], "k--", label="Perfect", linewidth=1.5)

    colors = ["tab:red", "tab:blue", "tab:green", "tab:orange"]
    for (name, probs), color in zip(probabilities_dict.items(), colors):
        accs = []
        for lo, hi in zip(bins[:-1], bins[1:]):
            mask = (probs >= lo) & (probs < hi)
            accs.append(labels[mask].mean() if mask.sum() > 0 else float("nan"))
        ece = expected_calibration_error(probs, labels, n_bins)
        ax.plot(bin_centers, accs, marker="o", label=f"{name} (ECE={ece:.3f})", color=color)

    ax.set_xlabel("Mean predicted probability", fontsize=12)
    ax.set_ylabel("Fraction of positives", fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

## Baseline: Uncalibrated Model

We train a standard binary classifier with Ludwig's ECD architecture. The decoder is `mlp_classifier`, which supports calibration and MC Dropout. The baseline has no calibration enabled.

In [ ]:
from ludwig.api import LudwigModel

WINE_FEATURES = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
    "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide",
    "density", "pH", "sulphates", "alcohol",
]

def make_input_features():
    return [
        {"name": f, "type": "number", "preprocessing": {"normalization": "zscore"}}
        for f in WINE_FEATURES
    ]

baseline_config = {
    "model_type": "ecd",
    "input_features": make_input_features(),
    "output_features": [
        {
            "name": "quality",
            "type": "binary",
            "decoder": {
                "type": "mlp_classifier",
                "num_fc_layers": 1,
                "output_size": 64,
                "dropout": 0.1,
                # calibration: null  (default — no calibration)
            },
            "loss": {"type": "binary_weighted_cross_entropy"},
        }
    ],
    "combiner": {
        "type": "concat",
        "num_fc_layers": 2,
        "output_size": 128,
        "dropout": 0.1,
    },
    "trainer": {"epochs": 30, "learning_rate": 0.001, "batch_size": 128},
}

shutil.rmtree("./results/baseline", ignore_errors=True)

baseline_model = LudwigModel(config=baseline_config, logging_level=logging.WARNING)
baseline_model.train(
    dataset=df,
    experiment_name="uncertainty",
    model_name="baseline",
    output_directory="./results/baseline",
)
print("Baseline training complete.")

## Evaluate Calibration

A well-calibrated model should have predicted probability ≈ empirical frequency within each confidence bin. The **reliability diagram** visualises this: a perfectly calibrated model lies on the diagonal. The **Expected Calibration Error (ECE)** summarises miscalibration as a weighted average gap between confidence and accuracy across bins.

In [ ]:
_, baseline_preds, _ = baseline_model.predict(dataset=df)

baseline_probs = baseline_preds["quality_probability_True"].values
true_labels = df["quality"].values

baseline_ece = expected_calibration_error(baseline_probs, true_labels)
print(f"Baseline ECE: {baseline_ece:.4f}")
print(f"Predicted probability range: [{baseline_probs.min():.3f}, {baseline_probs.max():.3f}]")
print(f"Mean predicted probability: {baseline_probs.mean():.3f}  (base rate: {true_labels.mean():.3f})")

plot_reliability_diagram(
    {"Baseline": baseline_probs},
    true_labels,
    title="Reliability Diagram — Baseline",
)

### What to look for

- Points **above the diagonal**: the model is underconfident in that bin (true frequency > predicted probability)
- Points **below the diagonal**: the model is overconfident (true frequency < predicted probability)
- Most neural networks are overconfident, especially in the high-confidence region (right side of the plot)

## Temperature Scaling

Temperature scaling is the simplest and most effective post-hoc calibration method. It learns a single scalar **T** on the validation set such that:

```
calibrated_logit = logit / T
```

- **T > 1**: softens the distribution (reduces overconfidence)
- **T < 1**: sharpens the distribution
- **T = 1**: no change

Critically, temperature scaling **never changes argmax predictions** — accuracy is unchanged. It only adjusts the probabilities.

In Ludwig, enable it by setting `calibration: temperature_scaling` in the decoder config.

In [ ]:
import copy

calibrated_config = copy.deepcopy(baseline_config)
# Enable temperature scaling — this is the only change from the baseline config
calibrated_config["output_features"][0]["decoder"]["calibration"] = "temperature_scaling"

print("Calibrated decoder config:")
import json
print(json.dumps(calibrated_config["output_features"][0]["decoder"], indent=2))

In [ ]:
shutil.rmtree("./results/calibrated", ignore_errors=True)

calibrated_model = LudwigModel(config=calibrated_config, logging_level=logging.WARNING)
calibrated_model.train(
    dataset=df,
    experiment_name="uncertainty",
    model_name="calibrated",
    output_directory="./results/calibrated",
)
print("Temperature scaling training complete.")

In [ ]:
_, calibrated_preds, _ = calibrated_model.predict(dataset=df)

calibrated_probs = calibrated_preds["quality_probability_True"].values
calibrated_ece = expected_calibration_error(calibrated_probs, true_labels)
print(f"Baseline ECE:              {baseline_ece:.4f}")
print(f"Temperature Scaling ECE:   {calibrated_ece:.4f}")

# Compare reliability diagrams
plot_reliability_diagram(
    {
        "Baseline": baseline_probs,
        "Temperature Scaling": calibrated_probs,
    },
    true_labels,
    title="Reliability Diagram — Baseline vs Temperature Scaling",
)

Temperature scaling should move the reliability curve closer to the diagonal without affecting classification accuracy. The ECE should decrease.

## MC Dropout

Monte Carlo Dropout treats dropout at **inference time** as approximate Bayesian inference. Instead of doing one deterministic forward pass, we do **T stochastic passes** (dropout enabled) and compute:

- **Mean probability** → the prediction
- **Variance across passes** → the uncertainty estimate

High-uncertainty samples are those where the model gives very different answers across passes — it genuinely does not know.

In Ludwig, enable MC Dropout via `mc_dropout_samples` on the decoder. The `uncertainty` column is automatically added to prediction outputs.

**Important**: You need `dropout > 0` in the decoder (and ideally in the combiner too) for MC Dropout to produce meaningful variance.

In [ ]:
mc_config = copy.deepcopy(baseline_config)
mc_config["output_features"][0]["decoder"].update({
    "dropout": 0.3,            # higher dropout → more variance across MC passes
    "mc_dropout_samples": 20,  # run 20 stochastic forward passes at inference
})
mc_config["combiner"]["dropout"] = 0.2  # combiner dropout also contributes to variance

print("MC Dropout config:")
print("  decoder:", json.dumps(mc_config["output_features"][0]["decoder"], indent=4))
print("  combiner dropout:", mc_config["combiner"]["dropout"])

In [ ]:
shutil.rmtree("./results/mc_dropout", ignore_errors=True)

mc_model = LudwigModel(config=mc_config, logging_level=logging.WARNING)
mc_model.train(
    dataset=df,
    experiment_name="uncertainty",
    model_name="mc_dropout",
    output_directory="./results/mc_dropout",
)
print("MC Dropout training complete.")

In [ ]:
_, mc_preds, _ = mc_model.predict(dataset=df)

print("MC Dropout output columns:")
print([c for c in mc_preds.columns if "quality" in c])

In [ ]:
if "quality_uncertainty" in mc_preds.columns:
    uncertainty = mc_preds["quality_uncertainty"].values
    mc_probs = mc_preds["quality_probability_True"].values

    print("Uncertainty statistics:")
    print(f"  mean  = {uncertainty.mean():.5f}")
    print(f"  std   = {uncertainty.std():.5f}")
    print(f"  max   = {uncertainty.max():.5f}")
    print(f"  p80   = {np.percentile(uncertainty, 80):.5f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram of uncertainty
    ax1.hist(uncertainty, bins=40, edgecolor="white", color="tab:green")
    ax1.set_xlabel("Uncertainty (variance across MC samples)")
    ax1.set_ylabel("Count")
    ax1.set_title("MC Dropout Uncertainty Distribution")

    # Uncertainty vs predicted probability
    scatter = ax2.scatter(
        mc_probs, uncertainty, c=true_labels, cmap="RdYlGn",
        alpha=0.4, s=10, vmin=0, vmax=1
    )
    plt.colorbar(scatter, ax=ax2, label="True label")
    ax2.set_xlabel("Mean predicted probability")
    ax2.set_ylabel("Uncertainty")
    ax2.set_title("Uncertainty vs Probability (coloured by true label)")

    plt.tight_layout()
    plt.show()
else:
    print("'quality_uncertainty' not found in predictions.")
    print("Available columns:", list(mc_preds.columns))

In [ ]:
if "quality_uncertainty" in mc_preds.columns:
    # Samples the model is most uncertain about
    top_k = 10
    top_uncertain_idx = np.argsort(uncertainty)[-top_k:][::-1]

    uncertain_df = df.iloc[top_uncertain_idx][["quality"] + WINE_FEATURES[:4]].copy()
    uncertain_df["predicted_prob"] = mc_probs[top_uncertain_idx].round(3)
    uncertain_df["uncertainty"] = uncertainty[top_uncertain_idx].round(5)
    print(f"Top-{top_k} highest-uncertainty samples:")
    print(uncertain_df.to_string())

### Interpreting the uncertainty plot

- **High uncertainty near p=0.5**: expected — the model genuinely cannot decide
- **High uncertainty near p=0 or p=1**: concerning — the model is confident but inconsistent across MC passes
- **Low uncertainty throughout**: dropout rate may be too low, or `mc_dropout_samples` too small

## Summary Comparison

In [ ]:
baseline_acc = (
    baseline_preds["quality_predictions"].astype(bool) == true_labels.astype(bool)
).mean()
calibrated_acc = (
    calibrated_preds["quality_predictions"].astype(bool) == true_labels.astype(bool)
).mean()
mc_acc = (
    mc_preds["quality_predictions"].astype(bool) == true_labels.astype(bool)
).mean()

mc_ece = expected_calibration_error(mc_preds["quality_probability_True"].values, true_labels)

print("=" * 60)
print(f"{'Model':<25} {'Accuracy':>10} {'ECE':>10}")
print("-" * 60)
print(f"{'Baseline':<25} {baseline_acc:>10.3%} {baseline_ece:>10.4f}")
print(f"{'Temperature Scaling':<25} {calibrated_acc:>10.3%} {calibrated_ece:>10.4f}")
print(f"{'MC Dropout':<25} {mc_acc:>10.3%} {mc_ece:>10.4f}")
print("=" * 60)
print()
print("Temperature scaling should preserve accuracy while reducing ECE.")

## When to Use Each Method

| | Temperature Scaling | MC Dropout |
|---|---|---|
| **Goal** | Better-calibrated probabilities | Per-sample uncertainty estimates |
| **Changes accuracy?** | No | Slightly (mean over passes) |
| **Inference cost** | None (just divides logits) | ~T× slower (T = mc_dropout_samples) |
| **Needs validation set?** | Yes (to learn T) | No |
| **Works without dropout?** | Yes | No — dropout must be > 0 |
| **Config key** | `decoder.calibration: temperature_scaling` | `decoder.mc_dropout_samples: N` |
| **Output columns** | `quality_probability_*` (rescaled) | + `quality_uncertainty` |
| **Good for** | Risk scoring, decision thresholds, ranking | Active learning, human-in-the-loop, safety-critical review |

### Can you use both?

Yes! Temperature scaling and MC Dropout are orthogonal. You can combine them:

```python
decoder_config = {
    "type": "mlp_classifier",
    "dropout": 0.3,
    "calibration": "temperature_scaling",  # calibrated mean probability
    "mc_dropout_samples": 20,              # + uncertainty estimate
}
```

This gives you both well-calibrated probabilities **and** per-sample uncertainty — useful when both the probability value and the model's confidence in that value matter.